### import libraries

In [1]:
import pymongo
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, date
import os
import pprint

load_dotenv(override=True)

True

### setup connection

In [2]:
mongo_uri = os.getenv("mongo_uri")
# mongo_uri_local = os.getenv("mongo_uri_local")

try:
    client = pymongo.MongoClient(mongo_uri)
except NameError:
    print("mongo_uri not available, trying mongo_uri_local")
    client=pymongo.MongoClient(mongo_uri_local)

from pymongo.uri_parser import parse_uri
# print(parse_uri(mongo_uri_local)["options"])
# print(repr(mongo_uri_local))



client.list_database_names()
db = client["klapp-prod"]

In [ ]:
list(db.list_collection_names())

#### search for document where different segments are noted

In [ ]:
list(db["label"].aggregate([{"$sample": {"size": 2}}]))

- We have deviating names for school_type and type in school.
- It seems like we also do not have standardized naming convention for different class types, there is no Kindergarten, Primarschule, Oberstufe, etc.

In [ ]:
db["school_types"].distinct("name")

In [ ]:
db["school"].distinct("type")

In [ ]:
db["label"].distinct("name")

#### pipe for different school segments (KiGa., Sek., etc.)

In [ ]:
print("Schulen mit '' als type: ", db["school"].count_documents({"type": ""}))
print("Schulen mit 'None' als type: ", db["school"].count_documents({"type": None}))
print("Schulen Total: ", db["school"].count_documents({}))

### try out $facet
- $facet lets you do sub pipelines on the same input document and bundles the output into one output document

- Das funktioniert nicht, da man nicht dem gleichen key zwei values zu weisen kann, das wäre das selbe wie  a = 2, a = 1 -> es würde es einfach überschreiben.


In [ ]:
segment_pipe = [{
    "$match": {
        "type": {"$eq": ""},
        "type": {"$eq": None}
    }},
    {"$count": "Schulen mit None oder ''"}
    ]

In [ ]:
list(db["school"].aggregate(segment_pipe))

- Deshalb kommt hier $facet in spiel

In [ ]:
facet_empty = [
    {"$facet": {
        "leerer_string": [
            {"$match": {"type": ""}},
            {"$count": "anzahl"}
        ],
        "none_wert": [
            {"$match": {"type": None}},
            {"$count": "anzahl"}
        ]
    }}
]

In [ ]:
print("Schulen mit '' und None als type: ", list(db["school"].aggregate(facet_empty)))

- Checke ob das None mit dem created datum zusammenhängt zwischen school und None

In [ ]:
none_pipe = [
    {"$facet": {
        "none_type": [
            {"$match": {
                "type": None,
                "invoicing_start_date": {"$ne": None}}},
            {"$project": {
                "created_at": 1,
                "invoicing_start_date": 1,
                "type": 1,
                "deleted_at": 1}},
        {"$limit": 5}
        ],
        "school_type": [
            {"$match": {
                "type": "school",
                "invoicing_start_date": {"$ne": None}}},
            {"$project": {
                "created_at": 1,
                "invoicing_start_date": 1,
                "type": 1,
                "deleted_at": 1}},
        {"$limit": 5}
        ]
    }},
    
]

- Es scheint keinen Zusammenhang zwischen invoicing_start_date, deleted_at und created_at zu type zu geben.

In [ ]:
pprint.pprint(list(db["school"].aggregate(none_pipe)))

- Herausfinden wann das type feature eingeführt wurde.

In [ ]:
pprint.pprint(list(db["migration"].find().sort("name", 1)))

#### Es macht keinen sinn über die Segmente zu trennen. Deshalb wird anhand der Schulgrösse geschaut, wie viele Lehrpersonen und Schüler registriert sind.

- Schauen ob es in user lehrpersonen gibt die `is_deleted:` True haben


In [ ]:
is_deleted_pipe = [
    {"$match": {
        "$or": [
            {"is_deleted": True},
            {"deleted_at": {"$ne": None}}],
        "role": "teacher"
    }},
    {"$count": "anzahl_gelöschte_lehrpersonen"}
]

In [ ]:
list(db["user"].aggregate(is_deleted_pipe))

- Da man bei migrations gesehen hat, dass is-deleted zu deleted-at eingeführt wurde. kurz checke ob entweder oder oder anderst.

In [ ]:
a = db["user"].count_documents({"is_deleted": True, "role": "teacher"})
b = db["user"].count_documents({"deleted_at": {"$ne": None}, "role": "teacher"})
u = db["user"].count_documents({
    "$or": [{"is_deleted": True}, {"deleted_at": {"$ne": None}}],
    "role": "teacher"
})
print(f"is_deleted={a}, deleted_at={b}, is_deleted OR deleted_at={u}")

- jeder deleted_at eintrag hat auch is deleted.

#### Generiere ersten Baustein: Lehrer pro Schule

- gelöschte accounts aber noch in einer Schule gelistet.

In [ ]:
teacher_per_school_pipe = [
    {"$match": {
        "is_deleted": True,
        "role": "teacher" 
    }},
    {"$lookup": {
        "from": "school",
        "localField": "_id",
        "foreignField": "teachers",
        "as": "school_match" 
    }},
    {"$match": {
        "school_match.0": {"$exists": True}
    }},
    {"$project": {
        "name": 1
    }},
    {"$count": "anzahl_lehrpersonen_gelöscht_aber_in_schule"}
]

In [ ]:
teacher_per_school = list(db["user"].aggregate(teacher_per_school_pipe))
teacher_per_school

- aktive Lehrpersonen pro Schule

In [ ]:
active_teachers = [
    {"$lookup": {
        "from": "user",
        "localField": "teachers",
        "foreignField": "_id",
        "as": "school_teachers"
    }},
    {"$project": {
        "name" :1,
        "anzahl_aktive_lehrer": {
        "$size": {
            "$filter": {
                "input": "$school_teachers",
                "as": "st",
                "cond": {"$ne": ["$$st.is_deleted", True]}
            }
        }
        }
    }},
    {"$sort": {
        "anzahl_aktive_lehrer": 1}}
]

In [ ]:
active_teachers_count = list(db["school"].aggregate(active_teachers))
active_teachers_count

In [ ]:
db["user"].distinct("role")

In [ ]:
aktive_schüler_pipe = [
    {"$match": {
        "deleted_at": {"$eq": None}
    }},
    {"$count": "anzahl_aktive_schüler"}
]

In [ ]:
aktive_schüler = list(db["student"].aggregate(aktive_schüler_pipe))
aktive_schüler

- kombinierte Pipe

In [ ]:
full_pipe = [
    {"$match": {
        "deleted_at": {"$eq": None}
    }},
    {"$lookup": {
        "from": "user",
        "localField": "teachers",
        "foreignField": "_id",
        "as": "school_teachers"
    }},
    {"$lookup": {
        "from": "student",
        "localField": "_id",
        "foreignField": "school",
        "as": "school_students"     ### muss anderen namen haben, da sonst dieses lookup das vorherige überschreibt
    }},
    {"$project": {
        "name" :1,
        "anzahl_aktive_lehrer": {
        "$size": {
            "$filter": {
                "input": "$school_teachers",
                "as": "st",
                "cond": {"$ne": ["$$st.is_deleted", True]}
                }
            }
        },
        "anzahl_aktive_schüler": {
            "$size": {
                "$filter": {
                    "input": "$school_students",
                    "as": "s",
                    "cond": {"$eq": ["$$s.deleted_at", None]}
                }
            }
        }
    }},
    {"$sort": {
        "anzahl_aktive_lehrer": -1}}
]

In [ ]:
combined = list(db["school"].aggregate(full_pipe))
combined

### In df laden und verteilung ansehen

In [ ]:
df = pd.DataFrame(combined)
df.describe()

- checke wie viele Schulen 0 lehrer und 0 schüler haben

In [ ]:
no_teachers =(df["anzahl_aktive_lehrer"] == 0).sum()
no_students = (df["anzahl_aktive_schüler"] == 0).sum()
no_teachers_and_no_students = ((df["anzahl_aktive_lehrer"] == 0) & (df["anzahl_aktive_schüler"] == 0)).sum()

print("Anzahl Schulen mit 0 Lehrer: ", no_teachers)
print("Anzahl Schulen mit 0 Schüler: ", no_students)
print("Rate der Schüler auf Lehrer -> 138 = gehören zusammen :::: ", no_teachers_and_no_students)
print("\n")
print("Ratio von keine lehrer und schüler", (no_teachers_and_no_students / no_teachers * 100).round(2))

In [ ]:
schulen_0_schueler_mit_lehrern = df[((df["anzahl_aktive_lehrer"] == 0 & df["anzahl_aktive_schüler"]) > 0)]
schulen_0_schueler_mit_lehrern = schulen_0_schueler_mit_lehrern.sort_values("anzahl_aktive_schüler", ascending=False)

schulen_0_schueler_mit_lehrern.to_csv("schulen_ohne_lehrer.csv")

### Pipes für Dashboard programmieren

- Aktive Schüler & Lehrer Pipe

In [ ]:
teacher_student_pipe = [
    {"$lookup": {
        "from": "user",
        "localField": "teachers",
        "foreignField": "_id",
        "as": "school_user"
    }},
    {"$lookup": {
        "from": "student",
        "localField": "_id",
        "foreignField": "school",
        "as": "school_students"     ### muss anderen namen haben, da sonst dieses lookup das vorherige überschreibt
    }},
    {"$project": {
        "name" :1,
        "anzahl_aktive_lehrer": {
        "$size": {
            "$filter": {
                "input": "$school_user",
                "as": "st",
                "cond": {"$ne": ["$$st.is_deleted", True]}
                }
            }
        },
        "anzahl_aktive_schüler": {
            "$size": {
                "$filter": {
                    "input": "$school_students",
                    "as": "s",
                    "cond": {"$eq": ["$$s.deleted_at", None]}
                }
            }
        },
        "anzahl_aktive_eltern": {
            "$size": {
                "$filter": {
                    "input": {"$ifNull": ["$school_user", []]},
                    "as": "u",
                    "cond": {"$ne": ["$$u.is_deleted", True]}
                }
            }
        },
        "created_at": 1,
        "loeschstatus": {
            "$cond":[{"$ne": ["$deleted_at", None]},
                  "$deleted_at",
                  None
                ]
            },
        "kuendigungsstatus": {
            "$cond": [{"$ne": ["$invoicing_cancellation_date", None]},
                      "$invoicing_cancellation_date",
                      None]
        },
        "status": {
            "$cond": [{"$eq": ["$deactivated", True]},
                      "Deaktiviert",
                      "Aktiv"]
        },
        "_id": 1
    }},
    # {"$sort": {
    #     "anzahl_aktive_lehrer": -1}}
    # {"$limit": 1}
]

In [ ]:
list(db["school"].aggregate(teacher_student_pipe))

In [ ]:
loeschstatus = list(db["school"].aggregate(teacher_student_pipe))

In [ ]:
df_loeschstatus = pd.DataFrame(loeschstatus)

df_loeschstatus[df_loeschstatus["loeschstatus"].notna()]

In [ ]:
db["school"].distinct("is_active")

- Absenz Pipe + joker Tage

In [ ]:
absence_pipe = [
    {"$match": {
        "event_type": {"$eq": "absence"},
        # "created_at": {"$gte": datetime(2026, 6, 1), "$lt": datetime(2026, 6, 30)}
    }},
    # {"$limit": 5},
    {"$group": {
        "_id": "$school",
        "anzahl_absenz": {"$sum": 1},
        "anzahl_joker_tage": {
            "$sum": {
                "$cond": [
                    {"$eq": ["$is_joker_day", True]},
                    1,
                    0
                ]
            }
        },
    }},
    {"$lookup": {
        "from": "school",
        "localField": "_id",
        "foreignField": "_id",
        "as": "school_info"
    }},
    {"$unwind": "$school_info"},
    {"$project": {
        "joker_tage_aktiviert": {
            "$cond": [
                {"$eq": ["$school_info.joker_days_enabled", True]},
                "Aktiviert",
                "Nicht Aktiviert"
    ]
},
        # "$school_info.client_name": 1
        "name": "$school_info.name",
        "anzahl_absenz": 1,
        "anzahl_joker_tage": 1,
        "_id": 1
    # }},
    # {"$sort": {
    #     "anzahl_absenz": -1
    }},
    # {"$sort": {
    #     "anzahl_joker_tage": 1,
    #     "joker_tage_aktiviert": 1}}
    
]

In [ ]:
db["school"].distinct("joker_days_enabled")

In [ ]:
absenz_pipe = list(db["event"].aggregate(absence_pipe))
absenz_pipe

- Nachrichten Pipe / davon Chat

In [ ]:
message_pipe = [
    {"$group": {
    "_id": "$school",
    "anzahl_notification": {"$sum": 1},
    "davon_chat_nachricht": {
    "$sum": {
        "$cond": [
            {"$eq": ["$is_chat", True]},
            1,
            0
        ]
    
    }
},
    }},
    {"$lookup": {
        "from": "school",
        "localField": "_id",
        "foreignField": "_id",
        "as": "school_notification"
    }},
    {"$unwind": "$school_notification"},
    {"$project": {
        "name": "$school_notification.name",
        "anzahl_notification": 1,
        "davon_chat_nachricht": 1,
        "_id": 1
    }}

]

In [ ]:
nachrichten_agg = list(db["notification"].aggregate(message_pipe))

In [ ]:
nachrichten_ext = nachrichten_agg
nachrichten_ext = sorted(nachrichten_ext, key=lambda x: (-x["anzahl_notification"], x["davon_chat_nachricht"]))
nachrichten_ext

- Event Pipe -> Termine, Meeting

In [ ]:
db["event"].distinct("event_type")

In [ ]:
event_pipe_meet = [
    {"$match": {
        "event_type": {"$eq": "meet"}
    }},
    {"$group": {
        "_id": "$school",
        "anzahl_meetings": {"$sum": 1}
    }},
    {"$lookup": {
        "from": "school",
        "localField": "_id",
        "foreignField": "_id",
        "as": "school_meet"
    }},
    {"$project": {
        "name": "$school_meet.name",
        "anzahl_meetings": 1,
        "_id": 1
    }}
]

In [ ]:
event_pipe_event = [
    {"$match": {
        "event_type": {"$eq": "event"}
    }},
    {"$group": {
        "_id": "$school",
        "anzahl_events": {"$sum": 1}
    }},
    {"$lookup": {
        "from": "school",
        "localField": "_id",
        "foreignField": "_id",
        "as": "school_event"
    }},
    {"$project": {
        "name": "$school_event.name",
        "anzahl_events": 1,
        "_id": 1
    }}
]

In [ ]:
anzahl_meetings = list(db["event"].aggregate(event_pipe_meet))
print(anzahl_meetings)

anzahl_event_event = list(db["event"].aggregate(event_pipe_event))
print(anzahl_event_event)

- Anzahl Dateien Pipe

In [ ]:
db["file"].distinct("recipient_classes.recipients")

In [ ]:
file_pipe = [
    {"$group": {
        "_id": "$school",
        "anzahl_dateien": {"$sum": 1}
    }},
    {"$lookup": {
        "from": "school",
        "localField": "_id",
        "foreignField": "_id",
        "as": "school_files"
    }},
    {"$project": {
        "name": "$school_files.name",
        "anzahl_dateien": 1,
        "_id": 1
    }}
]

In [ ]:
anzahl_files = list(db["file"].aggregate(file_pipe))
anzahl_files.sort(key=lambda x: x["anzahl_dateien"], reverse=False)
anzahl_files

- Umfragen Pipe

In [ ]:
question_pipe = [
    {"$group": {
        "_id": "$school",
        "anzahl_questions": {"$sum": 1}
    }},
    {"$lookup": {
        "from": "school",
        "localField": "_id",
        "foreignField": "_id",
        "as": "school_question"
    }},
    {"$project": {
        "name": "$school_question.name",
        "anzahl_questions": 1,
        "_id": 1
    }}
]

In [ ]:
anzahl_questions = list(db["question"].aggregate(question_pipe))
anzahl_questions

### Functions to simplyfy piping and df creation

In [ ]:
def df_creator(db, collection: str ,pipe):
    array = list(db[collection].aggregate(pipe))
    df = pd.DataFrame(array)
    return df

def merger(df_one, df_two, on, how):
    merged_df = pd.merge(df_one, df_two, on=on, how=how)
    return merged_df

def merge_alle(dataframes, on, how):
    result = dataframes[0]
    for df in dataframes[1:]:
        if "name" in df.columns:
            df = df.drop(columns="name")
        result = merger(result, df, on, how)
    return result

dfs = [("test1", "pipe1"), ("test2", "pipe2"), ("test3", "pipe3")]


### Das funktioniert nicht da [...] ausdrücke eine zahl -> index erwarten, zudem wurde merged_df nie zuge-
### wiesen.

# for i, name in enumerate(dfs):
#     df = df_creator(db, name, pipe)
#     merged = merger(df[i], df[i+1], on="_id", how="outer")
#     merged_df[name] = df[merged]

In [ ]:
pipeline_list = [
    (teacher_student_pipe, "school"), 
    (absence_pipe, "event"), 
    (message_pipe, "notification"), 
    (event_pipe_meet, "event"),
    (event_pipe_event, "event"),
    (file_pipe, "file"),
    (question_pipe, "question")]

all_df = []

for i, (pipes, collection) in enumerate(pipeline_list):
    df = df_creator(db, collection, pipes)
    all_df.append(df)

In [ ]:
[df.columns.tolist() for df in all_df]

In [ ]:
merged = merge_alle(all_df, on="_id", how="outer")

In [ ]:
merged.head()

In [ ]:
merged = merged.fillna(0).head()

In [ ]:
exception_cols = ["name", "joker_tage_aktiviert"]
digit_cols = merged.columns.difference(exception_cols)
merged[digit_cols] = merged[digit_cols].astype(int)

merged

### Test für Lukas, durchschnittlich empfangene Nachrichten in 90 Tagen pro Person

In [ ]:
pipe = [
    {"$match": {
        # "is_chat": False,
        "sent_at": {"$gte": datetime(2026, 2, 28), "$lt": datetime(2026, 3, 30)}
    }},
        {"$project": {
            "anzahl_eltern": {
                "$size": {
                    "$filter": {
                        "input": {"$ifNull": ["$recipient_users", []]},
                        "as": "e",
                        "cond": {"$eq": ["$$e.user_type", "parent"]}
                    }
                }
            }
        }},
    {"$group": {
        "_id": None,
        "total_nachrichten_an_eltern": {"$sum": "$anzahl_eltern"}
    }}
]

In [ ]:
anzahl_nachrichten = list(db["notification"].aggregate(pipe))

score = []
# for entry in enumerate(anzahl_nachrichten)



In [ ]:
anzahl_eltern = db["user"].count_documents({"role": "parent"})

In [ ]:
anzahl_nachrichten_zahl = anzahl_nachrichten[0]
anzahl_nachrichten_zahl = int(anzahl_nachrichten_zahl["total_nachrichten_an_eltern"])

mean_notifications = anzahl_nachrichten_zahl / anzahl_eltern


print(anzahl_nachrichten_zahl, anzahl_eltern, mean_notifications)

In [ ]:
db["notification"].count_documents({"recipient_users": {"$elemMatch": {"user_type": "parent"}}})

### Tests with datetime

In [ ]:
print(datetime(datetime.now().year, 8, 1).date())

In [ ]:
from datetime import datetime, timedelta

###########################################################################
HEUTE = datetime.now()
AKTUELLES_SCHULJAHR   = datetime(datetime.now().year, 7, 20)

VOR_30_TAGEN = HEUTE - timedelta(days=30)
VOR_30_TAGEN_PIPE = {"$gte": ["$created_at", VOR_30_TAGEN]}

VOR_90_TAGEN = HEUTE - timedelta(days=90)
VOR_90_TAGEN_PIPE = {"$gte": ["$created_at", VOR_90_TAGEN]}

LETZTES_SCHULJAHR_START = datetime(datetime.now().year - 1, 8, 1)
LETZTES_SCHULJAHR_ENDE = datetime(datetime.now().year, 7, 20)
LETZTES_SCHULJAHR_PIPE = {"$and": [{"$gte": ["$created_at", LETZTES_SCHULJAHR_START]}, {"$lte": ["$created_at", LETZTES_SCHULJAHR_ENDE]}]}

pipe_dict = {"30_tage": VOR_30_TAGEN_PIPE, "90_tage": VOR_90_TAGEN_PIPE, "letztes_schuljahr": LETZTES_SCHULJAHR_PIPE}

In [ ]:
pipe_dict.values()

In [ ]:
chat_pipe_dict = {}

for suffix, condition in pipe_dict.items():
    chat_pipe_dict[suffix] = {"$and": [condition, {"$eq": ["$is_chat", True]}]}

In [ ]:
chat_pipe_dict

#### Test with bexio

In [ ]:
pprint.pprint((db["invoices"].find_one()))

In [3]:
import sys
from pathlib import Path
from datetime import datetime, timedelta

projekt_root = Path.cwd().parent / "src"
sys.path.append(str(projekt_root.parent))

from src.lib import helpers, pipelines

###########################################################################
HEUTE = datetime.now()
AKTUELLES_SCHULJAHR   = datetime(datetime.now().year, 7, 20)

VOR_30_TAGEN = HEUTE - timedelta(days=30)
VOR_30_TAGEN_PIPE = {"$gte": ["$created_at", VOR_30_TAGEN]}

VOR_90_TAGEN = HEUTE - timedelta(days=90)
VOR_90_TAGEN_PIPE = {"$gte": ["$created_at", VOR_90_TAGEN]}

schuljahresende_dieses_jahr = datetime(HEUTE.year, 7, 20)

if HEUTE > schuljahresende_dieses_jahr:
    LETZTES_SCHULJAHR_START = datetime(HEUTE.year -1 , 8, 1)
    LETZTES_SCHULJAHR_ENDE = datetime(HEUTE.year , 7, 20)
else:
    LETZTES_SCHULJAHR_START = datetime(HEUTE.year -2, 8, 1)
    LETZTES_SCHULJAHR_ENDE = datetime(HEUTE.year -1, 7, 20)

# LETZTES_SCHULJAHR_START = datetime(datetime.now().year - 1, 8, 1)
# LETZTES_SCHULJAHR_ENDE = datetime(datetime.now().year, 7, 20)
LETZTES_SCHULJAHR_PIPE = {"$and": [{"$gte": ["$created_at", LETZTES_SCHULJAHR_START]}, {"$lte": ["$created_at", LETZTES_SCHULJAHR_ENDE]}]}
###########################################################################


pipe_dict = {"30_tage": VOR_30_TAGEN_PIPE, "90_tage": VOR_90_TAGEN_PIPE, "letztes_schuljahr": LETZTES_SCHULJAHR_PIPE}

money_pipe = [{
    "$group": {
        "_id": "$school_id",
        "anzahl_invoices_historie": {"$sum": "$amount"},
        **helpers.timeframe_fields("invoices", pipe_dict, is_money=True)
    }},
    {"$lookup": {
        "from": "school",
        "localField": "_id",
        "foreignField": "_id",
        "as": "school_total"
    }},
    {"$unwind": "$school_total"},
    {"$project": {
        "_id": 1,
        "anzahl_invoices_historie": 1,
        "invoicing_start_date": "$school_total.invoicing_start_date",
        "invoicing_cancellation_date": "$school_total.invoicing_cancellation_date",
        "anzahl_invoices_30_tage": 1,
        "anzahl_invoices_90_tage": 1,
        "anzahl_invoices_letztes_schuljahr": 1,
        "anzahl_invoices_historie": 1,
    }
}]

In [ ]:
# list(db["invoices"].aggregate(money_pipe))
# list(db["school"].aggregate(pipelines.teacher_student_pipe))[:5]

print(VOR_30_TAGEN)
print(HEUTE)
print(VOR_30_TAGEN)
print(VOR_90_TAGEN)

In [ ]:
db["event"].find_one({"event_type": "absence"}, sort=[("created_at", -1)])

- currently there arent any loeschstatus, so check if true

In [ ]:
loeschstatus_pipe = [{
    "$match": {
        "deleted_at": {"$ne": None}
    }},
    {"$count": "anzahl_mit_loeschdatum"
}]

In [ ]:
list(db["school"].aggregate(loeschstatus_pipe))

### To extract active parents I need to have a look a student document

In [ ]:
db["student"].find_one()

#### Some schools are setup as motherschool and daughter schools, the money amount gets summed into the mother school. Find out if there can be found a distinction and if it is possible to get the daughter school amount

In [ ]:
invoices_pipe = [{
    "$lookup": {
        "from": "school",
        "localField": "school_id",
        "foreignField": "_id",
        "as": "school_invoice"
    }},
    {"$match": {
        "school_invoice.name": {"$regex": "Herti"}
}}
]

In [ ]:
list(db["invoices"].aggregate(invoices_pipe))

In [ ]:
db["invoices"].find_one()

In [21]:
pipe_dict_date = helpers.build_pipe_dict("date", VOR_30_TAGEN, VOR_90_TAGEN, LETZTES_SCHULJAHR_START, LETZTES_SCHULJAHR_ENDE)

price_data_pipe = [
    {"$unwind": "$price_data"},
    {"$match": {
        "$expr": {"$ne": ["$price_data.school", "$school_id"]}
    }},
    {"$lookup": {
        "from": "school",
        "foreignField": "_id",
        "localField": "price_data.school",
        "as": "toechter_name"
    }},
    {"$unwind": "$toechter_name"},

    {"$facet": {
        "toechter_zu_mutter": [
            {"$group": {
                "_id": "$price_data.school",
                "mutterschule_id": {"$first": "$school_id"}
            }},
            {"$lookup": {
                "from": "school",
                "localField": "mutterschule_id",
                "foreignField": "_id",
                "as": "mother_school"
            }},
            {"$unwind": "$mother_school"},
            {"$project": {
                "mother_school_name": "$mother_school.name"
            }}
        ],
        "mutter_zu_toechter": [
            {"$group": {
                "_id": "$price_data.school",
                "mutterschule_id": {"$first": "$school_id"},
                "daughter_name": {"$first": "$toechter_name"},
                "toechter_betrag": {"$sum": "$price_data.amount_invoiced"}
            }},
            {"$group": {
                "_id": "$mutterschule_id",
                "anzahl_toechter": {"$sum": 1},
                "toechter_details": {"$push": {"name": "$daughter_name.name", "betrag": "$toechter_betrag"}}
            }},
            {"$lookup": {
                "from": "school",
                "localField": "_id",
                "foreignField": "_id",
                "as": "mother_school"
            }},
            {"$unwind": "$mother_school"},
            {"$project": {
                "mother_school_name": "$mother_school.name",
                "anzahl_toechter": 1,
                "toechter_details": 1
            }}
        ]
    }}
]

In [35]:
result = list(db["invoices"].aggregate(price_data_pipe))

In [ ]:
# invoice_list
result = list(db["invoices"].aggregate(price_data_pipe))



In [43]:
# invoice_list
invoice_dfs = list(db["invoices"].aggregate(pipelines.price_data_pipe))
df_toechter = pd.DataFrame(invoice_dfs[0]["toechter_zu_mutter"])
df_mutter = pd.DataFrame(invoice_dfs[0]["mutter_zu_toechter"])

### concat df's of invoice
df_concat = pd.concat([df_toechter, df_mutter])

In [48]:


df_concat["_id"] = df_concat["_id"].astype(str)
df_concat.dtypes

_id                       str
mother_school_name        str
anzahl_toechter       float64
toechter_details       object
dtype: object

#### A school has more than one Id?
--> those multiple mentiones were for each daughter school

In [ ]:
from bson import ObjectId

In [ ]:
pprint.pprint(db["school"].find_one({"_id": ObjectId("63720361e904f610cbe1971e")}))

In [ ]:
pprint.pprint(db["school"].find_one({"_id": ObjectId("6372031eb9f73f10cdc4a18f")}))

In [ ]:
pprint.pprint(db["school"].find_one({"_id": ObjectId("6399f02840de9610cd9f90a4")}))

#### check the state of mother_daughter.json to evaluate how to handle the nested toechter_details col.

In [9]:
df_invoice = pd.read_json("../data/mother_daughter.json")
type(df_invoice["toechter_details"])

pandas.Series